# 02.5 — Does the local mean-curvature estimator actually work? A Swiss roll sanity check

Stage 1 of this phase asks whether the local mean-curvature estimator can recover a
curvature field at all, before it is ever pointed at real embedding data. A stage-1 FAIL on
the real graph-of-function family has two possible causes: the data genuinely has nothing to
find, or the estimator is broken. Only a manifold whose curvature answer is known in advance
separates the two. The Swiss roll is that manifold here, exactly as it was for the two
decoder models before it.

This notebook carries no gate, no advance ratification of a threshold, and no threshold
table -- those live in a separate stage-1 document and a separate diagnostics runner. Nothing
printed below decides PASS or FAIL for the phase; it is a sanity check on the estimator's own
code, short enough to read in one sitting and cheap enough to re-run on a whim.

## §1. Setup

In [ ]:
import sys
from pathlib import Path

# import pu_manifold exactly as the other notebooks do -- relative, never from src/effdim/
NOTEBOOK_DIR = Path.cwd()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import make_swiss_roll

from pu_manifold import curvature_probe as cp

print(f"numpy {np.__version__}")
print(f"curvature_probe module: {cp.__file__}")

## §2. The Swiss roll

3,000 noiseless points. Coordinates are centred and divided by a single global scalar
standard deviation -- one number, `X_raw.std()` with no axis argument, so the shape is
preserved exactly and only the overall size changes. The colour in every plot below is the
roll's own arc-length parameter `t`, so the same colour always means the same place on the
sheet: colour bands staying in order is what makes a visual read of the curvature field
meaningful.

Curvature has units of inverse length, so scaling the point cloud by `1/global_std` scales
the true curvature by `global_std` in the opposite direction. `curvature_probe`'s
`swiss_roll_analytic_H_scaled` applies exactly that rescaling to the closed-form analytic
answer, so the ground truth is compared against the estimator in the same coordinates the
estimator actually sees.

In [ ]:
X_raw, t = make_swiss_roll(n_samples=3000, noise=0.0, random_state=20260807)
global_std = float(X_raw.std())
X = (X_raw - X_raw.mean(axis=0)) / global_std

print(f"X.shape     = {X.shape}")
print(f"global_std  = {global_std:.6f}")

## §3. The estimator

`curvature_probe.centroid_mean_curvature` is imported unchanged and called at `d=2`, the
roll's true intrinsic dimension (CLAUDE.md's mandate, and this phase's D-07: the working
dimension is never inherited from an earlier phase's frozen value). The neighbourhood size
`k=30` here is this notebook's own choice for a fast sanity read, and it is explicitly NOT
the gating `k` -- the phase's own stage-1 measurement grid fixes that value separately, and
nothing here pre-commits it.

In [ ]:
H_est_vec = cp.centroid_mean_curvature(X, k=30, d=2)
h_est = cp.mean_curvature_norm(H_est_vec)
h_true = cp.swiss_roll_analytic_H_scaled(t, global_std)

print(f"h_est.shape  = {h_est.shape}   h_true.shape = {h_true.shape}")
print(f"h_est  range = [{h_est.min():.4f}, {h_est.max():.4f}]")
print(f"h_true range = [{h_true.min():.4f}, {h_true.max():.4f}]")

## §4. Does the field track the truth, visually

Two checks. First, the roll in both required views -- a 3-D scatter (which hides the spiral
behind the default viewing angle) and the x-z plane (which shows it unambiguously) -- coloured
first by the analytic `||H||` and then by the estimated `||H||`, sharing one colour
normalization across the pair so the two are visually comparable. The estimated field should
show the same pattern as the analytic one: curvature highest at the roll's tight inner
turn (small `t`), falling off smoothly outward.

Second, the roll coloured by `t` itself (the colour-bands-stay-in-order check), alongside a
scatter of estimated against analytic `||H||` with the identity line drawn -- a slope away
from that line is where a systematic scale or convention error would show up.

In [ ]:
VIEW = dict(elev=12, azim=-78)  # looks along the roll's extrusion axis, so the spiral shows
vmin = float(min(h_true.min(), h_est.min()))
vmax = float(max(h_true.max(), h_est.max()))

fig = plt.figure(figsize=(11, 9))
for i, (values, title) in enumerate([(h_true, "analytic ||H||"), (h_est, "estimated ||H||")]):
    ax = fig.add_subplot(2, 2, i + 1, projection="3d")
    ax.scatter(X[:, 0], X[:, 1], X[:, 2], c=values, cmap="viridis", s=4, vmin=vmin, vmax=vmax)
    ax.view_init(**VIEW)
    ax.set_title(f"{title} (3-D)")
    ax.set_xlabel("x"); ax.set_ylabel("y"); ax.set_zlabel("z")

    ax2 = fig.add_subplot(2, 2, i + 3)
    ax2.scatter(X[:, 0], X[:, 2], c=values, cmap="viridis", s=4, vmin=vmin, vmax=vmax)
    ax2.set_title(f"{title} (x-z plane)")
    ax2.set_xlabel("x"); ax2.set_ylabel("z"); ax2.set_aspect("equal")
plt.tight_layout()
plt.show()

fig2 = plt.figure(figsize=(11, 4.5))
ax = fig2.add_subplot(1, 2, 1, projection="3d")
ax.scatter(X[:, 0], X[:, 1], X[:, 2], c=t, cmap="viridis", s=4)
ax.view_init(**VIEW)
ax.set_title("roll coloured by t (colour bands stay in order)")
ax.set_xlabel("x"); ax.set_ylabel("y"); ax.set_zlabel("z")

ax2 = fig2.add_subplot(1, 2, 2)
ax2.scatter(h_true, h_est, c=t, cmap="viridis", s=4)
lims = [float(min(h_true.min(), h_est.min())), float(max(h_true.max(), h_est.max()))]
ax2.plot(lims, lims, "k--", linewidth=1, label="identity")
ax2.set_xlabel("analytic ||H||"); ax2.set_ylabel("estimated ||H||")
ax2.set_title("estimated vs analytic, with the identity line")
ax2.legend(loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

## §5. How faithful is it, in numbers

Two numbers. Spearman rank correlation between the estimated and analytic `||H||` fields is
the statistic the phase actually gates on -- Phase 4 downstream consumes an ORDERING of
points by curvature, not the estimator's absolute scale, so ranking is what has to be right.
Median relative error is non-gating context: it says something about magnitude, but a
convention or scale mismatch could move it without moving the ranking at all.

In [ ]:
rho = cp.spearman_gate_statistic(h_est, h_true)
med_rel_err = cp.median_relative_error(h_est, h_true)

print(f"Spearman rho (h_est vs h_true) = {rho:.4f}")
print(f"median relative error          = {med_rel_err:.4f}")

## §6. Against a matched baseline

CLAUDE.md's usual baseline for this check is a matched plain autoencoder at the same
bottleneck -- but this estimator has no decoder, so that comparison does not apply. CLAUDE.md's
own carve-out for a no-decoder model is to compare against "a baseline that is known to
succeed." At `d=2` the local quadric ("bowl") fit needs only 3 quadratic coefficients and has
`k=30` neighbour samples to fit them from -- comfortably determined, unlike the same fit at the
phase's real working dimension of 20, where it needs 210 coefficients from far fewer samples.
That well-determined route is the known-good comparator used here: `quadric_mean_curvature`
and `estimator_agreement`, both imported unchanged, run alongside the gating estimator and the
substitution is recorded as a decision here rather than made silently.

In [ ]:
quadric_result = cp.quadric_mean_curvature(X, k=30, d=2)
agreement = cp.estimator_agreement(h_est, quadric_result["H_norm"])

print(f"quadric underdetermined   = {quadric_result['underdetermined']}")
print(f"quadric n_coefficients    = {quadric_result['n_coefficients']}")
print(f"agreement Spearman        = {agreement['agreement_spearman']:.4f}")
print(f"agreement median rel diff = {agreement['agreement_median_rel_diff']:.4f}")

## §7. Read-out

In [ ]:
pass_spearman = rho > 0.90
pass_median_rel_err = med_rel_err < 0.25
pass_agreement = agreement["agreement_median_rel_diff"] < 0.25
pass_determined = quadric_result["underdetermined"] is False

print(f"Spearman vs analytic H > 0.90                               : {pass_spearman}   (rho={rho:.4f})")
print(f"median relative error < 0.25                                : {pass_median_rel_err}   ({med_rel_err:.4f})")
print(f"agrees with the quadric baseline (median rel diff < 0.25)   : {pass_agreement}   ({agreement['agreement_median_rel_diff']:.4f})")
print(f"quadric baseline is determined at d=2 (not underdetermined) : {pass_determined}   ({not quadric_result['underdetermined']})")
print()

if pass_spearman and pass_median_rel_err and pass_agreement and pass_determined:
    print(
        "The centroid mean-curvature estimator recovers the Swiss roll's known curvature "
        "field -- ordered correctly, close in magnitude, and in agreement with an "
        "independently-derived, well-determined quadric fit at the same working dimension."
    )
else:
    print(
        "The centroid mean-curvature estimator did NOT clearly recover the Swiss roll's "
        "known curvature field on at least one of the checks above -- a real result about "
        "the estimator, not softened to pass."
    )
print()
print(
    "These four thresholds are sanity values chosen for this notebook alone; they are not "
    "the phase's gate, which is fixed only by the phase's own ratified pre-registration "
    "document."
)